<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c06-header.png" alt="Nextia Learning · Deep Learning and Transformers Explained" width="100%">

# How embeddings learn meaning: word2vec and GloVe

**[How embeddings learn meaning: word2vec and GloVe](https://learning.nextia-ai.com/courses/deep-learning/m08/word2vec-and-glove/)** · Deep Learning and Transformers Explained · Module 8 (optional), lesson 2 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** turn Larkfield's tickets into (centre, context) pairs, write and train skip-gram with negative sampling, read its nearest words honestly, compute PMI and an SVD of the co-occurrence matrix, and try analogies with pretrained GloVe vectors, including the ones that fail.

| | |
|---|---|
| **Time** | About 35 minutes, including about one minute of training |
| **Needs** | An internet connection for the first cells (about 2 MB of tickets and a 3.9 MB file of GloVe vectors). No account. Colab and Kaggle have PyTorch. |
| **You should know** | Embeddings and nearest words, from [Tokens and embeddings](https://learning.nextia-ai.com/courses/deep-learning/m05/tokens-and-embeddings/). |
| **You do not need** | Any earlier notebook of this course. |
| **Tested** | Python 3.12 with PyTorch 2.14.1 (CPU), in Jupyter on macOS, from a fresh start (*Restart and run all*); Colab and Kaggle have PyTorch preinstalled |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/deep-learning/m08/word2vec-and-glove/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C06/M08-L02-word2vec-and-glove/word2vec-and-glove-solution.ipynb).

## Setup: check PyTorch

Run the next cell. It uses the PyTorch that is already installed (Colab and Kaggle have it) or installs the CPU version if it is missing; that can take two minutes the first time. You should see a line that starts with `PyTorch 2.` and the device `cpu`. The course uses the CPU only, so a graphics card is not needed. 

In [ ]:
try:
    import torch
except ImportError:  # not installed: install the CPU version
    %pip install -q --disable-pip-version-check torch --index-url https://download.pytorch.org/whl/cpu
    import torch
import platform
torch.manual_seed(0)
print("PyTorch", torch.__version__, "| Python", platform.python_version(), "| device cpu")

## Setup: get the tickets and the course's code

Run the next cell. It downloads Larkfield's ticket texts (about 2 MB) and the course's `ticketnet.py` into a folder `ticket-router`, and checks each file's checksum. It needs an internet connection and no account, and takes about 10 seconds.

You should see `Ready: train 4696 tickets, valid 600, test 600.` If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `ticket-router` (in Colab: the **Files** panel on the left), then run the cell again.

The tickets are made up for this course by a language model, from a list of situations; no real customer wrote them.

In [ ]:
# Setup: download the tickets and ticketnet.py into ticket-router/ and check them.
import hashlib, os, subprocess, sys, urllib.request
from pathlib import Path

LABS = os.environ.get("NX_LABS_RAW", "https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/C06/")
FILES = {
    "train.csv": "98487169346f8b0d9bc55a3844849ab95d012b75f01d7083605a6a1cea57aa39",
    "valid.csv": "f0c7bfa75e75837cdaf891ac8d007a78a018ba8e3488155cf43f2c6a34a064c8",
    "test.csv": "f373f402bcc9742cdcf8829720710a97b7a1dce5d8e34aaa88d03e7e49a5404e",
    "split_manifest.json": "d043cf720697054fb16e6cfff72ecb49b471d75038308ad5836779a0dd0bf8a5",
}
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
PROJECT = Path("ticket-router").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:  # Python without certificates (python.org on macOS): try curl
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / "data" / name
    if not path.exists() or hashlib.sha256(path.read_bytes()).hexdigest() != expected:
        download(LABS + "data/" + name, path)
        if hashlib.sha256(path.read_bytes()).hexdigest() != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder ticket-router and run again.")
download(LABS + "project/ticketnet.py", PROJECT / "ticketnet.py")
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))
import ticketnet
from ticketnet import *  # TEAMS, Vocab, tokenize, the three networks, train(), evaluate() ...
train_rows, valid_rows, test_rows = (load_split(f"data/{p}.csv") for p in ("train", "valid", "test"))
print(f"Ready: train {len(train_rows)} tickets, valid {len(valid_rows)}, test {len(test_rows)}.")

## Setup: helpers

The next cell defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.
- `show_shape(name, tensor)` prints a tensor's shape, as the lessons do.

Run the cell. It prints nothing.

In [ ]:
# Helpers: check an answer without showing it.
import hashlib, json, warnings
import numpy as np
import pandas as pd
pd.set_option("display.precision", 4)
pd.set_option("display.max_columns", 30)
warnings.filterwarnings("ignore", category=FutureWarning)

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

torch.set_num_threads(2)  # the numbers in the lessons were measured on a CPU

def fingerprint_tensor(t):
    return fingerprint([round(float(v), 4) for v in t.flatten().tolist()])

def show_shape(name, t):
    print(f"{name:<12} shape {tuple(t.shape)}")

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. Build the pairs

Skip-gram uses the tokens of the course vocabulary (seen at least twice; the rare ones are dropped). Each token is a centre word in turn, and every token within 5 positions of it is a context word. The window never crosses from one ticket into the next.

Run the cell.

In [ ]:
import math, time
import numpy as np
import torch
import torch.nn.functional as F

vocab = Vocab([r["text"] for r in train_rows])
words = vocab.itos[2:]                          # the 2,862 real tokens (no <pad>, <unk>)
W2I = {w: i for i, w in enumerate(words)}
N = len(words)
sents = [[W2I[t] for t in tokenize(r["text"]) if t in W2I] for r in train_rows]
counts = np.bincount(np.concatenate(sents), minlength=N).astype(np.float64)
WINDOW, K, DIM = 5, 5, 50
print(N, "tokens in the vocabulary;", int(counts.sum()), "tokens in the training tickets")

tokens = tokenize("Can't log in and the password reset email never arrived. Please help.")
c = tokens.index("password")
print("context of 'password':", tokens[max(0, c - WINDOW):c] + tokens[c + 1:c + 1 + WINDOW])

> **Your turn.** Make the list `pairs` of all (centre, context) pairs of `tokens`, as tuples of two strings. Predict first: how many pairs does this 16-token ticket give?

In [ ]:
pairs = ...

In [ ]:
expect('the number of pairs', len(pairs) if isinstance(pairs, list) else None, 130)

Words near the start or the end of a ticket have fewer than 10 context words, so 16 tokens give 130 pairs, not 160.

## 2. The noise distribution

Negative samples come from the counts raised to the power 0.75 and scaled to add up to 1. The power lowers the frequent words and raises the rare ones.

> **Your turn.** Compute `p_noise`, an array of 2,862 probabilities. Then compare it with the plain shares `counts / counts.sum()` for "the" and "umbrella".

In [ ]:
p_noise = ...

In [ ]:
expect("p_noise of 'the'", round(float(p_noise[W2I['the']]), 5) if isinstance(p_noise, np.ndarray) else None, 0.02378)

## 3. The model

Two embedding tables: `v` for centre words and `u` for context words. The score of a pair is a dot product; the loss is −ln σ(u_o·v_c) − Σ ln σ(−u_k·v_c). `F.logsigmoid(x)` computes ln σ(x). The function `make_pairs` builds one epoch of pairs and drops frequent words at random (subsampling), as word2vec does.

Run the cell.

In [ ]:
keep_prob = np.minimum(1.0, np.sqrt(1e-3 / (counts / counts.sum())))
if not isinstance(p_noise, np.ndarray):  # your turn in section 2 is not done yet: use the reference
    print("p_noise has no value yet: this cell uses the reference counts ** 0.75 / sum, so the notebook can go on.")
    p_noise = counts ** 0.75 / (counts ** 0.75).sum()
noise = torch.tensor(p_noise, dtype=torch.float)

def make_pairs(rng):
    centres, contexts = [], []
    for s in sents:
        s = [t for t in s if rng.random() < keep_prob[t]]       # subsampling
        for i, c in enumerate(s):
            for j in range(max(0, i - WINDOW), min(len(s), i + WINDOW + 1)):
                if j != i:
                    centres.append(c)
                    contexts.append(s[j])
    return torch.tensor(centres), torch.tensor(contexts)

class SkipGram(torch.nn.Module):
    def __init__(self, n, dim):
        super().__init__()
        self.v = torch.nn.Embedding(n, dim)   # centre words
        self.u = torch.nn.Embedding(n, dim)   # context words
        torch.nn.init.uniform_(self.v.weight, -0.5 / dim, 0.5 / dim)
        torch.nn.init.zeros_(self.u.weight)

    def forward(self, c, o, neg):            # c, o: (B,)  neg: (B, K)
        vc = self.v(c)                                              # (B, d)
        pos = (self.u(o) * vc).sum(-1)                              # (B,)
        negs = torch.bmm(self.u(neg), vc.unsqueeze(-1)).squeeze(-1)  # (B, K)
        return (-F.logsigmoid(pos) - F.logsigmoid(-negs).sum(-1)).mean()

torch.manual_seed(0)
rng = np.random.default_rng(0)
sg = SkipGram(N, DIM)
show_shape("v table", sg.v.weight)
show_shape("u table", sg.u.weight)

> **Predict.** The `u` table starts at zero. What is the loss of any batch before training? Write it as a formula in K first.

> **Your turn.** The cell builds the first pairs and 5 negatives for each of the first 512 pairs (with separate random generators, so the training below stays the same as the lesson's). Compute `start_loss`, the loss of `sg` on these 512 pairs, as a Python float. Use `torch.no_grad()` and `.item()`.

In [ ]:
c0, o0 = make_pairs(np.random.default_rng(1))
neg0 = torch.multinomial(noise, 512 * K, replacement=True, generator=torch.Generator().manual_seed(1)).view(512, K)
start_loss = ...

In [ ]:
expect('the loss before training', round(start_loss, 4) if isinstance(start_loss, float) else None, 4.1589)

Every score is 0, every σ is 0.5, and each of the 1 + K terms is ln 2: the loss is (1 + K) ln 2 = 6 ln 2 = 4.159. If your first loss is far from this, a sign or a sum is wrong.

## 4. Train for 3 epochs

The lesson trained 8 epochs; 3 are enough to see the result here. Each epoch builds new pairs (subsampling drops different tokens), shuffles them, and runs the five training steps on batches of 512 pairs. It takes about 10 to 40 seconds, depending on the computer.

> **Predict.** The loss starts at 4.159. Where will it be after one epoch?

In [ ]:
opt = torch.optim.Adam(sg.parameters(), lr=0.002)
t0 = time.time()
losses = []
for epoch in range(1, 4):
    c_all, o_all = make_pairs(rng)
    perm = torch.randperm(len(c_all))
    c_all, o_all = c_all[perm], o_all[perm]
    total, steps = 0.0, 0
    for i in range(0, len(c_all), 512):
        neg = torch.multinomial(noise, len(c_all[i:i + 512]) * K, replacement=True).view(-1, K)
        loss = sg(c_all[i:i + 512], o_all[i:i + 512], neg)
        opt.zero_grad()
        loss.backward()
        opt.step()
        total += loss.item()
        steps += 1
    losses.append(round(total / steps, 3))
    print(f"epoch {epoch}: {len(c_all):,} pairs, mean loss {total / steps:.3f}")
print(f"{time.time() - t0:.0f} s")

In [ ]:
expect('the mean loss of epoch 3', losses[-1], 2.363, tolerance=0.02)

> **Check.** The lesson's run: 2.638, 2.414, 2.363 for the first three epochs (8 epochs end at 2.293). Another computer can give slightly different numbers.

## 5. Nearest words

Scale every row of `v` to length 1; then a dot product is a cosine similarity.

Run the cell.

In [ ]:
V = sg.v.weight.detach().numpy().astype(np.float64)
U = sg.u.weight.detach().numpy().astype(np.float64)
Vn = V / np.linalg.norm(V, axis=1, keepdims=True)

def nearest(word, M=Vn, k=5):
    sims = M @ M[W2I[word]]
    order = [i for i in np.argsort(-sims) if i != W2I[word]][:k]
    return [(words[i], round(float(sims[i]), 2)) for i in order]

for w in ["password", "card", "late"]:
    print(f"{w:<9} {int(counts[W2I[w]]):>5}x", nearest(w))

Read the lists honestly. "password" and "card" get sensible neighbours: words that appear in the same kind of ticket. "late" appears only 47 times, so a few tickets decide its vector, and its neighbours are mostly noise.

## 6. The co-occurrence matrix and PMI

Count how often each pair of tokens appears within 5 positions of each other. Then PMI compares each count with what the two words' frequencies alone predict: PMI = ln(X_ij · T / (X_i · X_j)).

Run the cell (about 10 seconds).

In [ ]:
X = np.zeros((N, N))
for s in sents:
    for i, c in enumerate(s):
        for j in range(max(0, i - WINDOW), i):
            X[c, s[j]] += 1
            X[s[j], c] += 1
row = X.sum(1)
T = X.sum()
with np.errstate(divide="ignore"):
    PMI = np.log(X * T / np.outer(row, row))
print(f"{(X > 0).mean():.2%} of the cells are not zero; T = {int(T):,}")
print("password-reset:", int(X[W2I['password'], W2I['reset']]), "times, PMI", round(PMI[W2I['password'], W2I['reset']], 2))
print("refund-delivery:", int(X[W2I['refund'], W2I['delivery']]), "times, PMI", round(PMI[W2I['refund'], W2I['delivery']], 2))

> **Your turn.** Levy and Goldberg showed that skip-gram's dot products v_c · u_o approximate PMI − ln K. Check it for your 3-epoch model: compute `dot_pr`, the dot product of `V[password]` and `U[reset]`, and `pmi_pr`, PMI − ln 5 for the same pair. Then compute `r`, the correlation of all `V[a] @ U[b]` with `PMI[a, b] - math.log(K)` over the pairs of the first 300 frequent alphabetic words (seen at least 20 times) that meet at least 5 times. Use `np.corrcoef`.

In [ ]:
dot_pr = ...
pmi_pr = ...
r = ...

In [ ]:
expect('PMI - ln 5 of password and reset', round(pmi_pr, 2) if isinstance(pmi_pr, float) else None, 2.79)

In [ ]:
expect('the correlation r is at least 0.75', (r >= 0.75) if isinstance(r, float) else None, True)

The lesson's 8-epoch model gives r = 0.852 over 4,949 pairs; after 3 epochs the correlation is already high. Skip-gram quietly fits the shifted PMI matrix.

## 7. A third method: SVD of shifted PPMI

Keep only max(PMI − ln 5, 0), then let the singular value decomposition find 50 directions. Each word gets 50 numbers, W = U₅₀ √S₅₀. No neural network, no learning rate.

Run the cell (10 to 30 seconds).

In [ ]:
SPPMI = np.maximum(np.nan_to_num(PMI, neginf=0.0) - math.log(K), 0)
Us, S, _ = np.linalg.svd(SPPMI, full_matrices=False)
W_svd = Us[:, :50] * np.sqrt(S[:50])
Wn = W_svd / np.linalg.norm(W_svd, axis=1, keepdims=True).clip(1e-8)
show_shape("W_svd", torch.tensor(W_svd))
for w in ["password", "card", "late"]:
    print(f"{w:<9}", nearest(w, Wn))

Compare these lists with skip-gram's. The two methods often agree on the frequent, specific words, because they fit the same statistics.

## 8. Pretrained GloVe and analogies

The next cell downloads a subset of GloVe 6B: 50 numbers for 20,367 words (3.9 MB), and checks its checksum. GloVe was trained by Jeffrey Pennington, Richard Socher and Christopher D. Manning (2014, *GloVe: Global Vectors for Word Representation*, EMNLP) on 6 billion tokens of Wikipedia 2014 and Gigaword 5. Stanford publishes the vectors under the Open Data Commons Public Domain Dedication and License (PDDL) 1.0.

Run the cell.

In [ ]:
GLOVE = Path("data/glove6b-50d-subset.npz")
if not GLOVE.exists() or hashlib.sha256(GLOVE.read_bytes()).hexdigest() != "da7efd39587326e233c16ffa8a8a89802f11de507bebaef1bb5b857d1f895f30":
    download(LABS + "data-glove/glove6b-50d-subset.npz", GLOVE)
glove_sha = hashlib.sha256(GLOVE.read_bytes()).hexdigest()
d = np.load(GLOVE)
gwords, G = d["words"].tolist(), d["vectors"]
gi = {w: k for k, w in enumerate(gwords)}
Gn = G / np.linalg.norm(G, axis=1, keepdims=True)
print(G.shape)

In [ ]:
expect('the checksum of the GloVe file', glove_sha, 'da7efd39587326e233c16ffa8a8a89802f11de507bebaef1bb5b857d1f895f30')

> **Your turn.** Write `analogy(a, b, c, k=3)`: the k words whose unit vectors have the largest cosine with a − b + c (use the unit vectors `Gn` for a, b and c, and scale the target to length 1). Leave out the three query words: set their similarity to `-np.inf`. Return a list of (word, cosine rounded to 3 decimals).

In [ ]:
def analogy(a, b, c, k=3):
    ...

print(analogy("king", "man", "woman"))

In [ ]:
expect_hash('the answer of king - man + woman', (analogy("king", "man", "woman") or [[None]])[0][0], 'ec08e28c75a45549')

Now try analogies that work and analogies that fail. Run the cell.

In [ ]:
for q in [("paris", "france", "italy"), ("berlin", "germany", "japan"), ("paris", "france", "canada"),
          ("brother", "boy", "girl"), ("refund", "money", "parcel"), ("doctor", "man", "woman")]:
    answer = analogy(*q)
    print(" - ".join(q[:2]), "+", q[2], "->", answer if answer is not None else "write analogy() above first")

Rome and Tokyo come first, but Ottawa does not appear (Vancouver and Toronto do), and "sister" loses to "cousin". "refund − money + parcel" gives nothing like "delivery": analogies fail quietly.

The last line is a **bias**: "doctor − man + woman → nurse" is a gender stereotype that the vectors learned from their text. Embeddings carry the biases of the text they learn from. Test pretrained vectors for the biases that matter to your task before you use them.

## 9. Failure case: forget to leave out the query words

> **Predict.** Without the `-np.inf` line, what is the top answer to king − man + woman?

In [ ]:
t = Gn[gi["king"]] - Gn[gi["man"]] + Gn[gi["woman"]]
sims = Gn @ (t / np.linalg.norm(t))
print([(gwords[n], round(float(sims[n]), 3)) for n in np.argsort(-sims)[:3]])

> **Check.** The top answer is "king" itself (0.885), queen second (0.852). The target is built from "king", so it stays close to it. Always leave the query words out, and say so when you report analogy results.

## Your turn: change K

> **Your turn.** With K = 15 negatives, what is the loss before training? Compute `start_loss_15` with a new `SkipGram` and the same first 512 pairs, with 15 negatives each.

In [ ]:
start_loss_15 = ...

In [ ]:
expect('the loss before training with K = 15', round(start_loss_15, 2) if isinstance(start_loss_15, float) else None, 11.09)

16 terms of ln 2 each: 11.09. The optimum also moves: v·u = PMI − ln 15, so only pairs that meet more than 15 times as often as chance get a positive score.

## Next

You trained skip-gram, checked it against PMI, built SVD vectors and tested pretrained GloVe, including its failures and its bias. The [lesson page](https://learning.nextia-ai.com/courses/deep-learning/m08/word2vec-and-glove/) has the explorers, the charts and the knowledge checks.

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [The mathematics of attention](https://learning.nextia-ai.com/courses/deep-learning/m08/the-mathematics-of-attention/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/deep-learning/m08/word2vec-and-glove/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The tickets are made up for this course by a language model; no real customer wrote them. GloVe 6B: Pennington, Socher and Manning (2014), PDDL 1.0.